# AIVRP EXPERIMENTS

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import math

def create_euclidean_graph(
    n: int,
    density: str,
    undirected_density: float = 0.50,
    penalty_pct: float = 0.75,
    seed: int | None = None,
) -> nx.DiGraph:

    rng = np.random.default_rng(seed)
    coords = rng.random((n, 2))

    UG_full = nx.Graph()
    for idx, (x, y) in enumerate(coords):
        UG_full.add_node(idx, pos=(x, y))
    for i in range(n):
        for j in range(i + 1, n):
            d = float(np.linalg.norm(coords[i] - coords[j]))
            UG_full.add_edge(i, j, weight=round(d, 4))

    if density == "dense":
        UG = UG_full
    elif density == "sparse":
        # start from the MST so the graph stays connected, then add random edges
        UG = nx.minimum_spanning_tree(UG_full, weight="weight")
        max_e = n * (n - 1) // 2
        target_e = int(np.ceil(undirected_density * max_e))
        candidates = list(set(UG_full.edges()) - set(UG.edges()))
        rng.shuffle(candidates)
        need = max(0, target_e - UG.number_of_edges())
        for (u, v) in candidates[:need]:
            UG.add_edge(u, v, weight=UG_full[u][v]["weight"])
    else:
        raise ValueError("density must be 'sparse' or 'dense'")

    assign_priority_weights(UG)

    DG = nx.DiGraph()
    for u, data in UG.nodes(data=True):
        DG.add_node(u, **data)
    for u, v, d in UG.edges(data=True):
        w = d["weight"]
        p = d["priority"]
        pen = round(penalty_pct * w, 4)
        DG.add_edge(u, v, weight=w, penalty=pen, priority=p)
        DG.add_edge(v, u, weight=w, penalty=pen, priority=p)

    return DG

def assign_priority_weights(G: nx.DiGraph) -> None:
    """Priority = edge betweenness scaled to integers 1-10."""
    bet = nx.edge_betweenness_centrality(G, normalized=True, weight="weight")
    vals = np.array(list(bet.values()), dtype=float)
    range_ = np.ptp(vals)
    if range_ == 0:
        scaled = np.zeros_like(vals)
    else:
        scaled = (vals - vals.min()) / range_

    ints = (1 + 9 * scaled).round().astype(int)

    for (u, v), pr in zip(bet, ints):
        G[u][v]["priority"] = int(pr)

def show_graph(G: nx.DiGraph, title: str = "") -> None:
    pos = {n: G.nodes[n]["pos"] for n in G}
    plt.figure(figsize=(4, 4))
    nx.draw_networkx_nodes(G, pos, node_size=250, node_color="#1f78b4")
    nx.draw_networkx_edges(G, pos, arrows=True, width=0.8)
    nx.draw_networkx_labels(G, pos, font_size=8, font_color="white")
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [13]:
def create_graph_from_vrp(
    path: str,
    penalty_pct: float = 0.75,
    undirected_density: float = 0.3,
    seed: int | None = None
) -> nx.DiGraph:
    coords = {}
    with open(path, 'r') as f:
        lines = f.readlines()

    section = None
    for line in lines:
        line = line.strip()
        if not line or line.startswith("EOF"):
            continue
        if "NODE_COORD_SECTION" in line:
            section = "coords"
            continue
        elif "DEMAND_SECTION" in line or "DEPOT_SECTION" in line:
            break

        if section == "coords":
            parts = line.split()
            node_id = int(parts[0]) - 1
            x, y = float(parts[1]), float(parts[2])
            coords[node_id] = (x, y)

    xs = [x for x, y in coords.values()]
    ys = [y for x, y in coords.values()]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)

    for node in coords:
        x, y = coords[node]
        coords[node] = (
            (x - min_x) / (max_x - min_x + 1e-6),
            (y - min_y) / (max_y - min_y + 1e-6)
        )

    n = len(coords)

    UG_full = nx.Graph()
    for idx in range(n):
        UG_full.add_node(idx, pos=coords[idx])
    for i in range(n):
        for j in range(i + 1, n):
            d = float(np.hypot(*(np.array(coords[i]) - np.array(coords[j]))))
            UG_full.add_edge(i, j, weight=round(d, 4))

    rng = np.random.default_rng(seed)

    UG = nx.minimum_spanning_tree(UG_full, weight="weight")
    max_edges = n * (n - 1) // 2
    target_edges = int(np.ceil(undirected_density * max_edges))
    candidates = list(set(UG_full.edges()) - set(UG.edges()))
    rng.shuffle(candidates)
    extra_needed = max(0, target_edges - UG.number_of_edges())
    for (u, v) in candidates[:extra_needed]:
        UG.add_edge(u, v, weight=UG_full[u][v]["weight"])

    assign_priority_weights(UG)

    DG = nx.DiGraph()
    for u, data in UG.nodes(data=True):
        DG.add_node(u, **data)
    for u, v, d in UG.edges(data=True):
        w = d["weight"]
        p = d["priority"]
        pen = round(penalty_pct * w, 4)
        DG.add_edge(u, v, weight=w, penalty=pen, priority=p)
        DG.add_edge(v, u, weight=w, penalty=pen, priority=p)

    return DG

In [ ]:
import dimod

def build_qubo(G: nx.DiGraph, K: int, budget: int, P1: float, P2: float):
    arcs, nodes = list(G.edges()), list(G.nodes())

    C_max     = max(G[u][v]['weight'] + G[u][v]['penalty'] for u, v in arcs)
    bits_dual = math.ceil(math.log2(C_max + 1))
    bits_s    = math.ceil(math.log2(budget + 1))

    var_index, idx = {}, 0

    # ------- x_{ij}
    for arc in arcs:
        var_index[('x', arc)] = idx; idx += 1

    # ------- pi_j (j != 0)
    for j in nodes:
        if j == 0: continue
        for l in range(bits_dual):
            var_index[('pi', j, l)] = idx; idx += 1

    # ------- lambda_nk (n != 0)
    for n in nodes:
        if n == 0: continue
        for k in range(K):
            for l in range(bits_dual):
                var_index[('lam', n, k, l)] = idx; idx += 1

    # ------- mu_k, nu_k
    for k in range(K):
        for l in range(bits_dual):
            var_index[('mu', k, l)] = idx; idx += 1
            var_index[('nu', k, l)] = idx; idx += 1

    # ------- two sided slack r_plus / r_minus
    for (i, j) in arcs:
        for k in range(K):
            for l in range(bits_dual):
                var_index[('r_plus',  i, j, k, l)] = idx; idx += 1
                var_index[('r_minus', i, j, k, l)] = idx; idx += 1

    # ------- global budget slack s
    for l in range(bits_s):
        var_index[('s_global', l)] = idx; idx += 1

    # ------- build QUBO
    Q = {i: {} for i in range(idx)}
    def add(u, v, c):
        if u > v: u, v = v, u
        Q[u][v] = Q[u].get(v, 0.0) + c

    # dual reward
    for j in nodes:
        if j == 0: continue
        for l in range(bits_dual):
            u = var_index[('pi', j, l)]
            add(u, u, -2**l)
    for k in range(K):
        for l in range(bits_dual):
            add(var_index[('mu', k, l)], var_index[('mu', k, l)], -2**l)
            add(var_index[('nu', k, l)], var_index[('nu', k, l)], -2**l)

    # flow penalty
    for (i, j) in arcs:
        c, d = G[i][j]['weight'], G[i][j]['penalty']
        xi   = var_index[('x', (i, j))]
        for k in range(K):
            terms = [(xi, d)]
            if j != 0:
                for l in range(bits_dual):
                    terms.append((var_index[('pi', j, l)], -2**l))
            if j != 0:
                for l in range(bits_dual):
                    terms.append((var_index[('lam', j, k, l)], -2**l))
            if i != 0:
                for l in range(bits_dual):
                    terms.append((var_index[('lam', i, k, l)],  2**l))
            if i == 0:
                for l in range(bits_dual):
                    terms.append((var_index[('mu', k, l)], -2**l))
            if j == 0:
                for l in range(bits_dual):
                    terms.append((var_index[('nu', k, l)], -2**l))
            for l in range(bits_dual):
                terms.append((var_index[('r_plus',  i, j, k, l)], -2**l))
                terms.append((var_index[('r_minus', i, j, k, l)],  2**l))

            for (u, w) in terms:
                add(u, u, P1 * (w*w + 2*c*w))
            for a in range(len(terms)):
                u, wu = terms[a]
                for v, wv in terms[a+1:]:
                    add(u, v, 2*P1*wu*wv)

    # budget penalty
    s_bits = [var_index[('s_global', l)] for l in range(bits_s)]
    terms  = [(var_index[('x', (i, j))], 1) for (i, j) in arcs] + \
             [(bit, 2**l) for l, bit in enumerate(s_bits)]
    for u, w in terms:
        add(u, u, P2 * (w*w - 2*budget*w))
    for a in range(len(terms)):
        u, wu = terms[a]
        for v, wv in terms[a+1:]:
            add(u, v, 2*P2*wu*wv)

    bqm = dimod.BinaryQuadraticModel.from_qubo(
        {(u, v): c for u, row in Q.items() for v, c in row.items()})
    return bqm, var_index

In [ ]:
def improved_qubo(bqm_base: dimod.BinaryQuadraticModel,
                  var_index: dict,
                  G: nx.DiGraph,
                  alpha: float = 5.0
                  ) -> dimod.BinaryQuadraticModel:
    bqm = bqm_base.copy()
    for (u, v) in G.edges():
        idx = var_index[('x', (u, v))]
        # 1) direct interdiction reward: subtract the true penalty d_{ij}
        d = G[u][v]['penalty']
        bqm.add_variable(idx, -d)
        # 2) priority reward: subtract alpha * p_ij
        p = G[u][v]['priority']
        bqm.add_variable(idx, -alpha * p)
    return bqm

In [ ]:
def choose_penalties(G: nx.DiGraph, K: int, safety_factor: float = 10.0):
    """
    Pick P1 and P2 so any single constraint breach costs more than
    flipping every interdiction bit could ever gain in the original QUBO.
    """
    arcs  = list(G.edges())

    C_max = max(G[u][v]["weight"] + G[u][v]["penalty"] for u,v in arcs)

    worst_orig_gain = len(arcs) * C_max

    P1 = math.ceil(safety_factor * worst_orig_gain * K)

    P2 = math.ceil(safety_factor * worst_orig_gain * len(arcs))

    return P1, P2

In [ ]:
def choose_scale_minbits(G, min_bits=5, max_bits=9):
    """
    Smallest power-of-two scale factor that gives min_bits <= bits_dual <= max_bits.
    """
    max_val = max(G[u][v]['weight'] + G[u][v]['penalty'] for u, v in G.edges())
    for exp in range(max_bits + 1):
        factor = 2 ** exp
        bits   = math.ceil(math.log2(max_val * factor + 1))
        if bits >= min_bits and bits <= max_bits:
            return factor, bits
    return 2 ** max_bits, max_bits

In [ ]:
def check_feasibility(sample, var_index, G, K, budget, verbose=True):
    inv = {idx: key for key, idx in var_index.items()}

    x_sum = sum(sample[idx] for idx, k in inv.items() if k[0] == 'x')
    s_val = sum(sample[idx]*2**inv[idx][-1] for idx in sample
                if inv[idx][0] == 's_global')
    budget_ok = (x_sum + s_val == budget)

    bits_dual = math.ceil(math.log2(
        max(G[u][v]['weight']+G[u][v]['penalty'] for u, v in G.edges()) + 1))

    flow_ok, first_bad = True, None
    for (i, j) in G.edges():
        for k in range(K):
            pi = sum(sample.get(var_index.get(('pi', j, l), -1), 0)*2**l
                     for l in range(bits_dual)) if j != 0 else 0
            lam_j = sum(sample.get(var_index.get(('lam', j, k, l), -1), 0)*2**l
                        for l in range(bits_dual)) if j != 0 else 0
            lam_i = sum(sample.get(var_index.get(('lam', i, k, l), -1), 0)*2**l
                        for l in range(bits_dual)) if i != 0 else 0
            mu = sum(sample.get(var_index.get(('mu', k, l), -1), 0)*2**l
                     for l in range(bits_dual)) if i == 0 else 0
            nu = sum(sample.get(var_index.get(('nu', k, l), -1), 0)*2**l
                     for l in range(bits_dual)) if j == 0 else 0
            r_plus  = sum(sample.get(var_index.get(('r_plus',  i, j, k, l), -1),0)*2**l
                          for l in range(bits_dual))
            r_minus = sum(sample.get(var_index.get(('r_minus', i, j, k, l), -1),0)*2**l
                          for l in range(bits_dual))
            lhs = (G[i][j]['weight']
                   + G[i][j]['penalty']*sample[var_index[('x', (i, j))]]
                   - pi - lam_j + lam_i - mu - nu
                   - r_plus + r_minus)
            if lhs != 0:
                flow_ok, first_bad = False, ((i, j), k, lhs)
                break
        if not flow_ok: break

    if verbose:
        print("\n---- Feasibility report ----")
        print(f"budget OK : {budget_ok}")
        print(f"flow   OK : {flow_ok}")
        if not flow_ok:
            arc, veh, res = first_bad
            print(f"  first flow imbalance on arc {arc}, veh {veh}, residual={res}")
        print("-----------------------------\n")
    return budget_ok and flow_ok

In [ ]:
def repair_to_feasible(sample: dict[int, int],
                       var_index: dict,
                       G: nx.DiGraph,
                       K: int) -> None:
    """
    Flip slack bits until every flow residual is zero.
    Handles the case where both r_plus & r_minus of a weight are already 1
    by turning the opposite bit off.
    Mutates sample in place.
    """
    bits_dual = math.ceil(math.log2(
        max(G[u][v]['weight'] + G[u][v]['penalty'] for u, v in G.edges()) + 1))

    while True:
        changed = False

        for (i, j) in G.edges():
            for k in range(K):

                c, d = G[i][j]['weight'], G[i][j]['penalty']
                x    = sample[var_index[('x', (i, j))]]

                def bit_sum(prefix, cond=True):
                    return sum(
                        sample.get(var_index.get(prefix+(l,), -1), 0) * (1 << l)
                        for l in range(bits_dual)
                    ) if cond else 0

                pi     = bit_sum(('pi',  j), j != 0)
                lam_j  = bit_sum(('lam', j, k), j != 0)
                lam_i  = bit_sum(('lam', i, k), i != 0)
                mu     = bit_sum(('mu',  k), i == 0)
                nu     = bit_sum(('nu',  k), j == 0)
                r_plus = bit_sum(('r_plus',  i, j, k))
                r_minus= bit_sum(('r_minus', i, j, k))

                gap = c + d*x - pi - lam_j + lam_i - mu - nu - r_plus + r_minus
                if gap == 0:
                    continue

                delta = abs(gap)
                for l in range(bits_dual):
                    if not (delta & (1 << l)):
                        continue

                    w        = 1 << l
                    key_plus = ('r_plus',  i, j, k, l)
                    key_minus= ('r_minus', i, j, k, l)
                    idx_plus = var_index[key_plus]
                    idx_minus= var_index[key_minus]

                    if gap > 0:
                        if sample[idx_plus] == 0:
                            sample[idx_plus] = 1
                        elif sample[idx_minus] == 0:
                            sample[idx_minus] = 1
                        else:
                            sample[idx_minus] = 0
                        gap -= w
                        changed = True

                    else:
                        if sample[idx_minus] == 0:
                            sample[idx_minus] = 1
                        elif sample[idx_plus] == 0:
                            sample[idx_plus] = 1
                        else:
                            sample[idx_plus] = 0
                        gap += w
                        changed = True

        if not changed:
            break

In [ ]:
import pandas as pd
import neal

def defender_cost(arcs, G):
    H = G.copy()
    for u, v in arcs:
        H[u][v]['weight'] += H[u][v]['penalty']
    return max(nx.single_source_dijkstra_path_length(H, 0, weight='weight').values())

def evaluate_metrics(G, K, B, alpha, num_reads=10, num_sweeps=1000):
    SCALE, bits_dual = choose_scale_minbits(G, min_bits=7, max_bits=9)
    for u, v in G.edges():
        G[u][v]['weight']  = int(round(G[u][v]['weight']  * SCALE))
        G[u][v]['penalty'] = int(round(G[u][v]['penalty'] * SCALE))

    P1, P2 = choose_penalties(G, K, safety_factor=5)
    bqm_base, vidx = build_qubo(G, K, B, P1, P2)
    bqm_imp = improved_qubo(bqm_base, vidx, G, alpha)
    sampler = neal.SimulatedAnnealingSampler()
    base_ss = sampler.sample(bqm_base, num_reads=num_reads, num_sweeps=num_sweeps).first
    mb = {k: int(v) for k, v in dict(base_ss.sample).items()}
    repair_to_feasible(mb, vidx, G, K)
    E_base = bqm_base.energy(mb)
    imp_ss = sampler.sample(bqm_imp, num_reads=num_reads, num_sweeps=num_sweeps).first
    mi = {k: int(v) for k, v in dict(imp_ss.sample).items()}
    repair_to_feasible(mi, vidx, G, K)
    E_imp_on_base = bqm_base.energy(mi)
    delta_q = 100 * (E_base - E_imp_on_base) / abs(E_base)
    arcs_base = [(u, v) for u, v in G.edges() if mb[vidx[('x', (u, v))]]]
    arcs_imp = [(u, v) for u, v in G.edges() if mi[vidx[('x', (u, v))]]]
    c_base = defender_cost(arcs_base, G)
    c_imp = defender_cost(arcs_imp, G)
    return delta_q, c_base, c_imp

def quality_vs_graph_size(node_list, K, B, reps=10, num_reads=50, num_sweeps=5000):
    energy_avg_records = []
    defender_records   = []
    print("---- Quality vs Graph Size ----")
    for n in node_list:
        eq_list = []
        bc_list = []
        ic_list = []
        for t in range(reps):
            density = "sparse" if t % 2 == 0 else "dense"
            G = create_euclidean_graph(n, density=density, seed=t)
            costs  = np.array([G[u][v]['weight']   for u, v in G.edges()])
            prios  = np.array([G[u][v]['priority'] for u, v in G.edges()])
            alpha  = costs.mean() / prios.mean()
            delta_q, c_base, c_imp = evaluate_metrics(G, K, B, alpha,
                                                 num_reads=num_reads,
                                                 num_sweeps=num_sweeps)
            eq_list.append(delta_q)
            bc_list.append(c_base)
            ic_list.append(c_imp)
            print("Nodes: " + str(n) + " Cycle: " + str(t) + " completed")
        energy_avg_records.append({"nodes": n, "ΔQ": np.mean(eq_list)})
        defender_records.append({"nodes": n,
                                 "base_cost": np.mean(bc_list),
                                 "imp_cost":  np.mean(ic_list)})
        print()
    return pd.DataFrame(energy_avg_records), pd.DataFrame(defender_records)

def quality_vs_vehicle_amount(node_size, fleet_list, B, reps=10, num_reads=50, num_sweeps=5000):
    energy_avg_records = []
    defender_records   = []
    print("\n---- Quality vs Vehicle Amount ----")
    for K in fleet_list:
        eq_list = []
        bc_list = []
        ic_list = []
        for t in range(reps):
            density = "sparse" if t % 2 == 0 else "dense"
            G = create_euclidean_graph(node_size, density=density, seed=t)
            costs  = np.array([G[u][v]['weight']   for u, v in G.edges()])
            prios  = np.array([G[u][v]['priority'] for u, v in G.edges()])
            alpha  = costs.mean() / prios.mean()
            delta_q, c_base, c_imp = evaluate_metrics(G, K, B, alpha,
                                                 num_reads=num_reads,
                                                 num_sweeps=num_sweeps)
            eq_list.append(delta_q)
            bc_list.append(c_base)
            ic_list.append(c_imp)
            print("Vehicles: " + str(K) + " Cycle: " + str(t) + " completed")
        energy_avg_records.append({"fleet": K, "ΔQ": np.mean(eq_list)})
        defender_records.append({"fleet": K,
                                 "base_cost": np.mean(bc_list),
                                 "imp_cost":  np.mean(ic_list)})
        print()
    return pd.DataFrame(energy_avg_records), pd.DataFrame(defender_records)

In [ ]:
import os

def test_set_A(folder_path, B=30, reps=1, num_reads=50, num_sweeps=5000, log_file="results.txt"):
    vrp_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".vrp")])
    total = len(vrp_files)

    with open(log_file, "a", encoding="utf-8") as log:
        for i, filename in enumerate(vrp_files):
            full_path = os.path.join(folder_path, filename)
            instance_name = os.path.splitext(filename)[0]

            try:
                parts = instance_name.lower().split('-')
                for p in parts:
                    if p.startswith('k'):
                        K = int(p[1:])
                        break
                else:
                    raise ValueError("No vehicle count found in filename.")
            except Exception as e:
                msg = f"Skipping {filename}: {e}"
                print(msg)
                log.write(msg + "\n")
                continue

            header = f"\n=== Running test case {i+1}/{total}: {filename} ===\n"
            header += f"  Vehicles (K): {K}\n"
            header += f"  Budget: {B}\n"
            header += f"  Tests remaining: {total - i - 1}\n"
            print(header)
            log.write(header)

            for t in range(reps):
                try:
                    log.write(f"Cycle {t+1} Loading graph\n")
                    print(f"Cycle {t+1} Loading graph")

                    G = create_graph_from_vrp(full_path)
                    costs = np.array([G[u][v]['weight'] for u, v in G.edges()])
                    prios = np.array([G[u][v]['priority'] for u, v in G.edges()])
                    alpha = costs.mean() / prios.mean()

                    log.write(f"Graph loaded. Starting evaluate_metrics...\n")
                    print(f"Graph loaded. Starting evaluate_metrics...")

                    delta_q, c_base, c_imp = evaluate_metrics(
                        G, K, B, alpha,
                        num_reads=num_reads,
                        num_sweeps=num_sweeps
                    )

                    result = f"Cycle {t+1}: ΔQ = {delta_q:.2f}% | Base = {c_base} | Improved = {c_imp}\n"
                    print(result)
                    log.write(result)

                except Exception as e:
                    error_msg = f"Cycle {t+1} failed: {e}\n"
                    print(error_msg)
                    log.write(error_msg)

In [ ]:
import matplotlib.ticker as mtick

def plot_results(df_energy_nodes, df_defender_nodes,
                 df_energy_fleet, df_defender_fleet):
    # Energy improvement vs graph size
    plt.figure()
    plt.plot(df_energy_nodes['nodes'], df_energy_nodes['ΔQ'], marker='o')
    plt.title('Energy Improvement vs Graph Size')
    plt.xlabel('# Nodes'); plt.ylabel('ΔQ (%)')

    # Defender cost vs graph size
    plt.figure()
    plt.plot(df_defender_nodes['nodes'], df_defender_nodes['base_cost'], label='Base')
    plt.plot(df_defender_nodes['nodes'], df_defender_nodes['imp_cost'], label='Improved')
    plt.title('Defender Cost vs Graph Size')
    plt.xlabel('# Nodes'); plt.ylabel('Cost'); plt.legend()

    # Energy improvement vs fleet size
    plt.figure()
    plt.plot(df_energy_fleet['fleet'], df_energy_fleet['ΔQ'], marker='o')
    plt.title('Energy Improvement vs Fleet Size')
    plt.xlabel('# Vehicles'); plt.ylabel('ΔQ (%)')
    plt.gca().xaxis.set_major_locator(mtick.MaxNLocator(integer=True))

    # Defender cost vs fleet size
    plt.figure()
    plt.plot(df_defender_fleet['fleet'], df_defender_fleet['base_cost'], label='Base')
    plt.plot(df_defender_fleet['fleet'], df_defender_fleet['imp_cost'], label='Improved')
    plt.title('Defender Cost vs Fleet Size')
    plt.xlabel('# Vehicles'); plt.ylabel('Cost'); plt.legend()
    plt.gca().xaxis.set_major_locator(mtick.MaxNLocator(integer=True))

    plt.show()

In [ ]:
import re

def plot_cvrplib_results(log_file="results.txt"):
    with open(log_file, 'r') as file:
        lines = file.readlines()

    instance_names = []
    base_costs = []
    improved_costs = []

    instance_pattern = re.compile(r'test case \d+/\d+: (.+?)\.vrp')
    base_pattern = re.compile(r'Base = (\d+)')
    improved_pattern = re.compile(r'Improved = (\d+)')

    for line in lines:
        instance_match = instance_pattern.search(line)
        base_match = base_pattern.search(line)
        improved_match = improved_pattern.search(line)

        if instance_match:
            instance_names.append(instance_match.group(1))
        if base_match:
            base_costs.append(int(base_match.group(1)))
        if improved_match:
            improved_costs.append(int(improved_match.group(1)))

    plt.figure(figsize=(8, 5))
    x = range(len(instance_names))

    plt.plot(x, base_costs, label='Base', color='blue', marker='o')
    plt.plot(x, improved_costs, label='Improved', color='orange', marker='o')

    plt.xticks(ticks=x, labels=instance_names, rotation=90, fontsize=8)
    plt.xlabel('CVRPLIB Instance')
    plt.ylabel('Defender Cost')
    plt.title('Defender Cost of Base vs Improved Model on CVRPLIB Instances')
    plt.legend()
    plt.tight_layout()
    plt.show()